# Palmer Penguins: Statistical Foundations

## 1. Dataset framing

**Project purpose:**  
To practice core statistical tests and simple linear regression using the Palmer Penguins dataset.

**Source and provenance:**  
Originally by researchers at the Palmer Station in Antarctica between 2007 and 2009. It contains physical measurements and other information for three species of penguins. This CSV is a simplified and curated version found on Github - it is not the original dataset.

**Unit of observation:**  
One row per penguin

**Sample:**  
A sample of 344 adult penguins from three species (Adelie, Chinstrap, and Gentoo) observed on three islands in the Palmer Archipelago.

**Study type:**  
Observational study because the data were collected without any intervention or manipulation.

**Scope and limitations:**  
- The data can be used to describe the sampled penguins and investigate
  differences and associations between their recorded characteristics.
- Because this is observational data, associations cannot by themselves
  establish causation.
- The sample is limited to particular species, islands, adult penguins
  and years. It therefore cannot automatically represent all penguins.
- The simplified dataset omits information available in the original
  raw dataset.

### First analysis question

Among the penguins recorded in this study, how does body mass differ between species?

- **Outcome variable:** body_mass_g, numeric
- **Grouping variable:** species, categorical

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

penguins_url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/master/inst/extdata/penguins.csv"
penguins_path = DATA_RAW / "penguins.csv"

if not penguins_path.exists():
    penguins = pd.read_csv(penguins_url)
    penguins.to_csv(penguins_path, index=False)
else:
    penguins = pd.read_csv(penguins_path)
    
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [2]:
print(penguins.shape)
print("=============")
print(penguins.columns.to_list())
print("=============")
penguins.info()

(344, 8)
['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
 7   year               344 non-null    int64  
dtypes: float64(4), int64(1), str(3)
memory usage: 21.6 KB


In [3]:
penguins.duplicated().sum()


np.int64(0)

In [4]:
penguins.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64

In [5]:
(penguins.isna().mean() * 100).round(2)

species              0.00
island               0.00
bill_length_mm       0.58
bill_depth_mm        0.58
flipper_length_mm    0.58
body_mass_g          0.58
sex                  3.20
year                 0.00
dtype: float64

In [6]:
missing_rows = penguins[penguins.isna().any(axis=1)]
missing_rows

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN,2007
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN,2007
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,NaN,2007
11,Adelie,Torgersen,37.8,17.3,180.0,3700.0,NaN,2007
47,Adelie,Dream,37.5,18.9,179.0,2975.0,NaN,2007
178,Gentoo,Biscoe,44.5,14.3,216.0,4100.0,NaN,2007
218,Gentoo,Biscoe,46.2,14.4,214.0,4650.0,NaN,2008
256,Gentoo,Biscoe,47.3,13.8,216.0,4725.0,NaN,2009
268,Gentoo,Biscoe,44.5,15.7,217.0,4875.0,NaN,2009


In [7]:
q1_data = penguins[["species", "body_mass_g"]].dropna()

print(q1_data.shape)
q1_data.isna().sum()

(342, 2)


species        0
body_mass_g    0
dtype: int64

In [9]:
q1_data["species"].value_counts(dropna=False)

species
Adelie       151
Gentoo       123
Chinstrap     68
Name: count, dtype: int64

In [14]:
q1_data["body_mass_g"].describe().round(2)

count     342.00
mean     4201.75
std       801.95
min      2700.00
25%      3550.00
50%      4050.00
75%      4750.00
max      6300.00
Name: body_mass_g, dtype: float64

In [20]:
species_summary = q1_data.groupby("species").agg(
    n=("body_mass_g", "count"),
    mean=("body_mass_g", "mean"),
    std=("body_mass_g", "std"),
    median=("body_mass_g", "median")
).round(2).reset_index()

species_summary

,species,n,mean,std,median
0,Adelie,151,3700.66,458.57,3700.0
1,Chinstrap,68,3733.09,384.34,3700.0
2,Gentoo,123,5076.02,504.12,5000.0
